In [ ]:
# Last Update: 6/29/2026 ; 6:13pm

In [2]:
# MULTIMODAL PREPROCESSING: EEG + fNIRS 
# Set the paths based on your system before running the code
# you can adjust which subjects to process at the bottom of this cell

import os
import numpy as np
import scipy.io as sio
from scipy.io import loadmat
from scipy.signal import butter, filtfilt, resample
from scipy.interpolate import griddata
import mne
import mne_icalabel
import pywt
import scipy.stats as stats
import pandas as pd
from scipy.interpolate import UnivariateSpline
import matplotlib.pyplot as plt  # NEW: needed for before/after preprocessing plots
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
#BASE_DATASET_PATH = r"D:\Research\EEG-fNIRS\project\project\Dataset"
#OUTPUT_PATH = r"D:\Research\EEG-fNIRS\project\project\Results\MULTIMODAL_PREPROCESSED"
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"
OUTPUT_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"


# window parameters
EEG_WINDOW_LEN = 3.0      
EEG_STEP = 1.0            # 66% overlap
EEG_EPOCH_TMIN = 0.0
# EEG_EPOCH_TMAX = 8.0
# FIX (Problem 2): the n-back task period is 40 s long (2 s instruction + 40 s task + 20 s rest,
# Shin et al. 2018). The previous value of 8 s only let the sliding window traverse the first
# 8 s of each trial, discarding ~80% of the task period and, more importantly, was being applied
# on top of STIMULUS-level markers (every ~2 s) rather than TRIAL-level markers, which produced
# massive window overlap across different stimuli instead of covering the full cognitive-load trial.
EEG_EPOCH_TMAX = 40.0      # NEW: cover the full 40 s task period of a single n-back trial

FNIRS_WINDOW_LEN = 10.0  
FNIRS_STEP = 1.0          # 90% overlap
FNIRS_EPOCH_TMIN = 0.0
FNIRS_EPOCH_TMAX = 18.0   # bacause of the hemodynamic delay
# NOTE: FNIRS_EPOCH_TMAX is no longer used directly to bound the slide range (the EEG trial length
# now governs how many windows are produced, see extract_synchronized_windows below), but it is left
# untouched here exactly as before so no other part of the code/structure changes.

SFREQ_EEG = 200.0
SFREQ_FNIRS = 10.0      

# NEW (Problem 1 fix support): explicit trial-level marker maps based on the official dataset
# documentation (Shin et al. 2018, Dataset_description_MATLAB.pdf):
#   EEG  mrk_nback.event.desc : 112 = 0-back session, 128 = 2-back session, 144 = 3-back session
#   fNIRS mrk_nback.event.desc:   7 = 0-back session,   8 = 2-back session,   9 = 3-back session
# These TRIAL-onset codes (one per ~40 s task period) replace the old STIMULUS-level codes
# (16/48/64/80/96 for EEG; 1/2/3/4/5 placeholders) that fired every ~2 s within a trial.
EEG_TRIAL_LABEL_MAP = {112: 0, 128: 1, 144: 2}      # NEW: EEG session/trial-onset codes -> class
FNIRS_TRIAL_LABEL_MAP = {7: 0, 8: 1, 9: 2}          # NEW: fNIRS session/trial-onset codes -> class


# LOAD EEG DATA
def load_eeg_data(eeg_folder):
    cnt_path = os.path.join(eeg_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(eeg_folder, 'mrk_nback.mat')
    
    print(f"Loading EEG from: {cnt_path}")
    
    # Load signal
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith('__')]
    cnt_key = real_cnt_keys[0]
    cnt_struct = mat_cnt[cnt_key]
    
    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]
    
    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6
    
    # making sure the sampling rate of 200
    if sfreq != SFREQ_EEG:
        print(f"Resampling from {sfreq} to {SFREQ_EEG} Hz")
        from scipy.signal import resample
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG
    
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)
    
    # EOG channels
    eog_channels = [ch for ch in ['VEOG', 'HEOG'] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: 'eog' for ch in eog_channels})
    
    raw.set_montage('standard_1005', match_case=False, on_missing='ignore')
    
    # Load markers 
    mat_mrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
    real_mrk_keys = [k for k in mat_mrk.keys() if not k.startswith('__')]
    mrk_key = real_mrk_keys[0]
    mrk_struct = mat_mrk[mrk_key]
    
    if hasattr(mrk_struct, 'time'):
        event_positions = np.round(mrk_struct.time * sfreq / 1000).astype(int)
    elif hasattr(mrk_struct, 'pos'):
        event_positions = np.array(mrk_struct.pos, dtype=int) - 1
    else:
        raise ValueError("Could not find 'time' or 'pos' in marker struct")
    
    # Event extraction
    if hasattr(mrk_struct, 'event') and hasattr(mrk_struct.event, 'desc'):
        desc = mrk_struct.event.desc
        if desc.ndim > 1:
            desc = desc.squeeze()
        event_codes = np.asarray(desc, dtype=int).flatten()
    elif hasattr(mrk_struct, 'y'):
        y = mrk_struct.y
        if y.ndim > 1:
            if y.shape[0] == 1 or y.shape[1] == 1:
                event_codes = y.flatten()
            else:
                event_codes = np.argmax(y, axis=0) + 1
        else:
            event_codes = y.flatten()
    else:
        event_codes = event_positions.copy()
    
    events = np.zeros((len(event_positions), 3), dtype=int)
    events[:, 0] = event_positions
    events[:, 2] = event_codes
    
    print(f"EEG loaded: {raw.n_times} samples, {sfreq} Hz, {len(events)} markers")
    print(f"unique markers: {np.unique(event_codes)}")
    
    return raw, events, sfreq


# LOAD AND PREPROCESS FNIRS DATA
def load_and_preprocess_fnirs(subject_id):
#    subject_num = subject_id[2:]
#    subject_path = os.path.join(BASE_DATASET_PATH, subject_num)
#    fnirs_folder = os.path.join(subject_path, f"{subject_id}-NIRS")
    fnirs_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-NIRS")
   
    cnt_path = os.path.join(fnirs_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(fnirs_folder, 'mrk_nback.mat')  # NEW: fNIRS trial markers needed for the quality-check plot and (optionally) downstream use
    mnt_path = os.path.join(fnirs_folder, 'mnt_nback.mat')
    
    if not os.path.exists(cnt_path):
        raise FileNotFoundError(f"fNIRS file not found: {cnt_path}")
    
    print(f"Loading raw fNIRS from: {cnt_path}")
    
    # cnt_data = loadmat(cnt_path)
    # cnt = cnt_data['cnt_nback'][0, 0]
    #
    # mnt_data = loadmat(mnt_path)
    # mnt = mnt_data['mnt_nback'][0, 0]
    # FIX (Problem 1): the lines above used the default scipy.io.loadmat() call (struct_as_record=True),
    # while the rest of the pipeline (load_eeg_data) consistently uses
    # struct_as_record=False, squeeze_me=True. Mixing the two access styles is fragile: under the
    # default mode, struct fields are accessed via numpy structured-array indexing (e.g. cnt['oxy'][0,0]['x']),
    # while under struct_as_record=False they are accessed as plain Python attributes (e.g. cnt.oxy.x).
    # Verified against the actual VP001 .mat structure (inspect_data.py output): cnt_nback has fields
    # ['oxy','deoxy'], each of which is itself a struct with field 'x' holding the (n_samples, 36) signal,
    # and mnt_nback has fields ['x','y','pos_3d','clab'] holding (36,) coordinate vectors. Using the
    # unified squeeze_me=True / struct_as_record=False loader removes the [0,0] nested-indexing style and
    # avoids a silent shape mismatch in spatial_mapping() if the indexing assumption is ever wrong.
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)  # NEW: unified loader (matches load_eeg_data)
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith('__')]     # NEW
    cnt = mat_cnt[real_cnt_keys[0]]                                          # NEW: cnt is now a MATLAB-struct object with .oxy / .deoxy attributes

    mat_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)  # NEW: unified loader (matches load_eeg_data)
    real_mnt_keys = [k for k in mat_mnt.keys() if not k.startswith('__')]     # NEW
    mnt = mat_mnt[real_mnt_keys[0]]                                          # NEW: mnt is now a MATLAB-struct object with .x / .y attributes

    # Load fNIRS trial markers (NEW: needed only for the before/after QC plot at the end of this
    # function; segmentation itself happens later using EEG_TRIAL_LABEL_MAP / FNIRS_TRIAL_LABEL_MAP
    # inside extract_synchronized_windows, which keeps this function's return signature unchanged).
    fnirs_events_time = None  # NEW
    fnirs_events_code = None  # NEW
    if os.path.exists(mrk_path):  # NEW
        mat_fmrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)  # NEW
        real_fmrk_keys = [k for k in mat_fmrk.keys() if not k.startswith('__')]    # NEW
        fmrk = mat_fmrk[real_fmrk_keys[0]]                                         # NEW
        if hasattr(fmrk, 'time') and hasattr(fmrk, 'event') and hasattr(fmrk.event, 'desc'):  # NEW
            fnirs_events_time = np.asarray(fmrk.time).flatten()                     # NEW
            fnirs_events_code = np.asarray(fmrk.event.desc).flatten()               # NEW
    
    # Extract actual sampling rate (making sure it's 10Hz)
    # sfreq_fnirs_raw = float(cnt['fs'][0, 0]) if 'fs' in cnt.dtype.names else 10.0
    # FIX (Problem 1, continued): 'fs' is a field of cnt.oxy / cnt.deoxy (each a sub-struct), not of
    # cnt itself (verified: cnt_nback.oxy.fs == 10, cnt_nback.deoxy.fs == 10 in the inspected VP001 file).
    # The original line would raise an AttributeError under the new loader (cnt has no .dtype/.fs), so
    # it must read fs from the oxy sub-struct.
    sfreq_fnirs_raw = float(cnt.oxy.fs) if hasattr(cnt, 'oxy') and hasattr(cnt.oxy, 'fs') else 10.0  # NEW
    print(f"raw fNIRS sampling rate: {sfreq_fnirs_raw} Hz")
    
    # Extract raw oxy/deoxy
    # if 'oxy' in cnt.dtype.names:
    #     oxy_raw = cnt['oxy'][0, 0]['x']
    #     deoxy_raw = cnt['deoxy'][0, 0]['x']
    # else:
    #     oxy_raw = cnt['oxy']
    #     deoxy_raw = cnt['deoxy']
    # FIX (Problem 1, continued): with struct_as_record=False, squeeze_me=True, the signal matrix is
    # accessed as cnt.oxy.x / cnt.deoxy.x (plain attribute access), matching the (n_samples, 36) shape
    # confirmed in inspect_data.py for VP001 (21178, 36).
    oxy_raw = np.asarray(cnt.oxy.x, dtype=np.float64)     # NEW
    deoxy_raw = np.asarray(cnt.deoxy.x, dtype=np.float64)  # NEW
    
    print(f"Raw fNIRS: oxy {oxy_raw.shape}, deoxy {deoxy_raw.shape}")
    
    # Convert to HbO/HbR
    print("Converting to HbO/HbR...")
    hbo_raw, hbr_raw = calculate_hbo_hbr(oxy_raw, deoxy_raw)
    
    # Wavelet denoising 
    print("Wavelet denoising...")
    hbo_wv = wavelet_denoising(hbo_raw)
    hbr_wv = wavelet_denoising(hbr_raw)
    
    # MARA on full signal
    print("MARA...")
    hbo_mara = apply_mara(hbo_wv, fs=sfreq_fnirs_raw)
    hbr_mara = apply_mara(hbr_wv, fs=sfreq_fnirs_raw)
    
    # Low-pass filter 
    # print("ow-pass filter (0.15 Hz)...")
    # hbo_filtered = lowpass_filter(hbo_mara, fs=sfreq_fnirs_raw, cutoff=0.15)
    # hbr_filtered = lowpass_filter(hbr_mara, fs=sfreq_fnirs_raw, cutoff=0.15)
    # FIX (Problem 4): 0.15 Hz was an unjustified, overly conservative cutoff with no clear basis in
    # the EEG-fNIRS cognitive-workload literature reviewed for this thesis (Saadati et al. 2020 used
    # 0.08 Hz; Bunterngchit et al. 2024 used 0.2 Hz; both target removal of cardiac (~1 Hz) and
    # respiratory (~0.2-0.3 Hz) physiological noise while preserving the slow hemodynamic response to
    # cognitive load). Aligning the cutoff to 0.2 Hz brings the pipeline in line with the closest
    # published precedent on this exact dataset (Bunterngchit et al., IEEE JTEHM 2024) while still
    # removing high-frequency physiological noise.
    FNIRS_LOWPASS_CUTOFF_HZ = 0.2  # NEW: justified cutoff matching Bunterngchit et al. (2024) on the same dataset
    print(f"Low-pass filter ({FNIRS_LOWPASS_CUTOFF_HZ} Hz)...")  # NEW
    hbo_filtered = lowpass_filter(hbo_mara, fs=sfreq_fnirs_raw, cutoff=FNIRS_LOWPASS_CUTOFF_HZ)  # NEW
    hbr_filtered = lowpass_filter(hbr_mara, fs=sfreq_fnirs_raw, cutoff=FNIRS_LOWPASS_CUTOFF_HZ)  # NEW
    
    # dynamic downsampling to 10 Hz
    print(f"Resampling from {sfreq_fnirs_raw} Hz to {SFREQ_FNIRS} Hz...")
    from scipy.signal import resample
    n_samples = int(hbo_filtered.shape[0] * SFREQ_FNIRS / sfreq_fnirs_raw)
    hbo_resampled = resample(hbo_filtered, n_samples, axis=0)
    hbr_resampled = resample(hbr_filtered, n_samples, axis=0)
    
    # spatial mapping to 16x16
    print("Spatial mapping to 16x16...")
    hbo_mapped = spatial_mapping(hbo_resampled, mnt)
    hbr_mapped = spatial_mapping(hbr_resampled, mnt)
    
    # Stack as channels: (n_samples, 16, 16, 2)
    fnirs_data = np.stack([hbo_mapped, hbr_mapped], axis=-1)
    
    print(f"fNIRS preprocessed: {fnirs_data.shape}")

    # NEW (user request): save a before/after preprocessing QC plot for fNIRS (one representative
    # channel, raw HbO vs fully preprocessed/resampled HbO). Saved per-subject next to that subject's
    # output folder. Does not alter any returned value or downstream logic.
    try:  # NEW
        qc_channel = 0  # NEW: first fNIRS channel, arbitrary but fixed for reproducibility
        t_raw = np.arange(oxy_raw.shape[0]) / sfreq_fnirs_raw  # NEW
        t_proc = np.arange(hbo_resampled.shape[0]) / SFREQ_FNIRS  # NEW
        fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)  # NEW
        axes[0].plot(t_raw, oxy_raw[:, qc_channel], color='tab:red', linewidth=0.6)  # NEW
        axes[0].set_title(f"{subject_id} - fNIRS RAW HbO (channel {qc_channel}, before preprocessing)")  # NEW
        axes[0].set_xlabel("Time (s)")  # NEW
        axes[0].set_ylabel("HbO (raw units)")  # NEW
        axes[1].plot(t_proc, hbo_resampled[:, qc_channel], color='tab:blue', linewidth=0.8)  # NEW
        axes[1].set_title(f"{subject_id} - fNIRS PREPROCESSED HbO (channel {qc_channel}, after wavelet+MARA+lowpass+resample)")  # NEW
        axes[1].set_xlabel("Time (s)")  # NEW
        axes[1].set_ylabel("HbO (mmol/L)")  # NEW
        plt.tight_layout()  # NEW
        qc_dir = os.path.join(OUTPUT_PATH, subject_id)  # NEW
        os.makedirs(qc_dir, exist_ok=True)  # NEW
        fig.savefig(os.path.join(qc_dir, f"{subject_id}_fnirs_before_after.png"), dpi=150)  # NEW
        plt.close(fig)  # NEW
        print(f"Saved fNIRS before/after QC plot for {subject_id}")  # NEW
    except Exception as e:  # NEW
        print(f"Could not save fNIRS QC plot for {subject_id}: {e}")  # NEW
    
    return fnirs_data

# fNIRS HELPER FUNCTIONS
def calculate_hbo_hbr(oxy, deoxy):
    lambda1, lambda2 = 690, 830
    L = 3.0
    
    def get_dpf(lmbda, age=25):
        alpha, beta, gamma = 223.3, 0.05624, 0.8493
        delta, eps, zeta = -5.723e-7, 0.001245, -0.9025
        return alpha + beta * (age ** gamma) + delta * (lmbda ** 3) + eps * (lmbda ** 2) + zeta * lmbda
    
    dpf1 = get_dpf(lambda1)
    dpf2 = get_dpf(lambda2)
    
    e1_hbo, e1_hbr = 0.3197, 1.3430
    e2_hbo, e2_hbr = 1.0559, 0.6924
    
    denom = L * (e1_hbr * e2_hbo - e2_hbr * e1_hbo)
    hbo = (e1_hbr * (deoxy / dpf2) - e2_hbr * (oxy / dpf1)) / denom
    hbr = (e1_hbo * (oxy / dpf1) - e2_hbo * (deoxy / dpf2)) / denom
    
    return hbo, hbr

def wavelet_denoising(data, alpha=0.1, wavelet='db5'):
    cleaned = np.zeros_like(data)
    n_samples, n_channels = data.shape
    
    for ch in range(n_channels):
        signal = data[:, ch]
        level = min(int(np.floor(np.log2(n_samples))), 6)
        coeffs = pywt.wavedec(signal, wavelet, level=level)
        new_coeffs = [coeffs[0]]
        for i in range(1, len(coeffs)):
            d_coeffs = coeffs[i].copy()
            sigma = np.median(np.abs(d_coeffs - np.median(d_coeffs))) / 0.6745
            if sigma > 0:
                threshold = sigma * stats.norm.ppf(1 - alpha / 2)
                d_coeffs[np.abs(d_coeffs) > threshold] = 0
            new_coeffs.append(d_coeffs)
        cleaned[:, ch] = pywt.waverec(new_coeffs, wavelet)[:n_samples]
    
    return cleaned

def apply_mara(data, fs=10.0, window_size=41, threshold_factor=1.8):
    n_samples, n_channels = data.shape
    corrected = np.zeros_like(data)
    edge_len = int(0.5 * fs)
    
    for ch in range(n_channels):
        signal = data[:, ch].copy()
        msd = pd.Series(signal).rolling(window=window_size, center=True, min_periods=1).std().to_numpy()
        threshold = np.median(msd) + (threshold_factor * np.std(msd))
        ma_mask = (msd > threshold).astype(int)
        
        diff_mask = np.diff(ma_mask, prepend=0, append=0)
        starts = np.where(diff_mask == 1)[0]
        ends = np.where(diff_mask == -1)[0]
        indices = sorted(list(set([0, n_samples] + list(starts) + list(ends))))
        segments = [signal[indices[i]:indices[i+1]].copy() for i in range(len(indices) - 1)]
        
        if not segments:
            corrected[:, ch] = signal
            continue
        
        final_parts = [segments[0]]
        for i in range(1, len(segments)):
            prev_seg = final_parts[-1]
            curr_seg = segments[i]
            val_prev = np.median(prev_seg[-edge_len:]) if len(prev_seg) >= edge_len else prev_seg[-1]
            val_curr = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
            shift = val_prev - val_curr
            
            if ma_mask[indices[i]] == 1:
                x = np.arange(len(curr_seg))
                s_val = max(np.var(curr_seg) * len(curr_seg), 1e-6)
                try:
                    spline = UnivariateSpline(x, curr_seg, s=s_val)
                    curr_seg = curr_seg - spline(x)
                    val_curr_new = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
                    curr_seg += (val_prev - val_curr_new)
                except:
                    curr_seg += shift
            else:
                if abs(shift) > 2 * np.std(msd):
                    curr_seg += shift
            final_parts.append(curr_seg)
        
        full_signal = np.concatenate(final_parts)[:n_samples]
        t_idx = np.arange(n_samples)
        original_trend = np.polyval(np.polyfit(t_idx, data[:, ch], 1), t_idx)
        corrected_trend = np.polyval(np.polyfit(t_idx, full_signal, 1), t_idx)
        full_signal += (original_trend - corrected_trend)
        corrected[:, ch] = full_signal
    
    return corrected

def lowpass_filter(data, fs=10.0, cutoff=0.15):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(3, normal_cutoff, btype='low')
    filtered = np.zeros_like(data)
    for ch in range(data.shape[1]):
        filtered[:, ch] = filtfilt(b, a, data[:, ch])
    return filtered

def spatial_mapping(data, mnt):
    n_samples, n_channels = data.shape
    # x_coords = mnt['x'].flatten()
    # y_coords = mnt['y'].flatten()
    # FIX (Problem 1, continued): with struct_as_record=False, squeeze_me=True (applied at the call
    # site in load_and_preprocess_fnirs), mnt is a MATLAB-struct object, so its fields are accessed
    # as attributes (mnt.x / mnt.y), not via dict-style indexing. mnt.x and mnt.y are each (36,)
    # coordinate vectors (verified in inspect_data.py: mnt_nback.x shape=(36,), mnt_nback.y shape=(36,)),
    # matching the 36 fNIRS channels. The previous dict-style access (mnt['x']) would raise a
    # TypeError under the new loader, which is exactly why the loader and this access style must be
    # changed together.
    x_coords = np.asarray(mnt.x).flatten()  # NEW: attribute access matching struct_as_record=False loader
    y_coords = np.asarray(mnt.y).flatten()  # NEW
    
    grid_x, grid_y = np.mgrid[min(x_coords):max(x_coords):16j, 
                               min(y_coords):max(y_coords):16j]
    points = np.column_stack((x_coords, y_coords))
    
    mapped = np.zeros((n_samples, 16, 16), dtype=np.float32)
    for t in range(n_samples):
        grid_z = griddata(points, data[t, :], (grid_x, grid_y), method='cubic', fill_value=0)
        mapped[t] = grid_z
    
    return mapped

# PREPROCESS EEG
def preprocess_eeg(raw, events):
    print("Preprocessing EEG...")

    # NEW (user request): keep a copy of the raw signal (one representative channel) for the
    # before/after preprocessing QC plot saved at the end of this function.
    raw_before_qc = raw.copy()  # NEW
    
    raw_filtered = raw.copy().notch_filter(freqs=50.0, picks='eeg', verbose=False)
    raw_filtered.filter(l_freq=1.0, h_freq=40.0, method='fir', verbose=False)
    
    # Detect bad channels
    raw_temp = raw_filtered.copy().filter(l_freq=10.0, h_freq=40.0, method='fir', verbose=False)
    eeg_picks = mne.pick_types(raw_temp.info, eeg=True, eog=False)
    eeg_names = [raw_temp.ch_names[idx] for idx in eeg_picks]
    
    temp_data = raw_temp.get_data(picks=eeg_picks)
    stds = np.std(temp_data, axis=1)
    median_std = np.median(stds)
    mad = np.median(np.abs(stds - median_std)) if np.median(np.abs(stds - median_std)) > 0 else 1e-6
    corr_matrix = np.corrcoef(temp_data)
    mean_corrs = np.mean(corr_matrix, axis=1)
    
    bad_channels = []
    for i, ch_name in enumerate(eeg_names):
        if ch_name.upper() in ['VEOG', 'HEOG', 'TP9']:
            continue
        if stds[i] > (median_std + 4.5 * mad) or stds[i] < (median_std - 4.5 * mad) or mean_corrs[i] < 0.30:
            bad_channels.append(ch_name)
    
    if bad_channels:
        print(f"Bad channels: {bad_channels}")
        raw_filtered.info['bads'] = bad_channels
    
    if raw_filtered.info['bads']:
        raw_filtered.interpolate_bads(reset_bads=True, verbose=False)
    
    raw_filtered.set_eeg_reference('average', projection=False, verbose=False)
    
    # ICA
    picks_for_ica = mne.pick_types(raw_filtered.info, eeg=True, eog=False, exclude='bads')
    corrected_rank = max(1, len(picks_for_ica) - 1)
    ica = mne.preprocessing.ICA(n_components=corrected_rank, method='infomax',
                                fit_params=dict(extended=True), random_state=42, verbose=False)
    ica.fit(raw_filtered, picks=picks_for_ica)
    
    ic_labels = mne_icalabel.label_components(raw_filtered, ica, method='iclabel')
    artifact_idx = [idx for idx, label in enumerate(ic_labels['labels']) 
                    if label in ['eye blink', 'eye movement', 'muscle artifact', 'line noise']]
    ica.exclude = artifact_idx
    if len(ica.exclude) > 0:
        ica.apply(raw_filtered, verbose=False)
        print(f"Removed {len(ica.exclude)} ICA components")
    
    # Case-insensitive channel drop
    channels_to_drop = [ch for ch in raw_filtered.ch_names if ch.upper() in ['VEOG', 'HEOG', 'TP9']]
    if channels_to_drop:
        raw_filtered.drop_channels(channels_to_drop)
        print(f"Dropped channels: {channels_to_drop}")

    # NEW (user request): save a before/after preprocessing QC plot for EEG (one representative
    # channel, raw vs fully preprocessed). Saved per-subject. Does not alter the returned object.
    try:  # NEW
        subject_id_for_plot = raw.info.get('subject_info', None)  # NEW (may be None, handled below)
        qc_ch_name = raw_before_qc.ch_names[0]  # NEW: first EEG channel, fixed for reproducibility
        qc_ch_name_after = qc_ch_name if qc_ch_name in raw_filtered.ch_names else raw_filtered.ch_names[0]  # NEW
        raw_seg_before = raw_before_qc.copy().pick([qc_ch_name]).get_data()[0]  # NEW
        raw_seg_after = raw_filtered.copy().pick([qc_ch_name_after]).get_data()[0]  # NEW
        t_before = np.arange(len(raw_seg_before)) / raw_before_qc.info['sfreq']  # NEW
        t_after = np.arange(len(raw_seg_after)) / raw_filtered.info['sfreq']  # NEW
        fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)  # NEW
        axes[0].plot(t_before, raw_seg_before, color='tab:red', linewidth=0.4)  # NEW
        axes[0].set_title(f"EEG RAW (channel {qc_ch_name}, before preprocessing)")  # NEW
        axes[0].set_xlabel("Time (s)")  # NEW
        axes[0].set_ylabel("Amplitude (V)")  # NEW
        axes[1].plot(t_after, raw_seg_after, color='tab:blue', linewidth=0.4)  # NEW
        axes[1].set_title(f"EEG PREPROCESSED (channel {qc_ch_name_after}, after notch+bandpass+CAR+ICA)")  # NEW
        axes[1].set_xlabel("Time (s)")  # NEW
        axes[1].set_ylabel("Amplitude (V)")  # NEW
        plt.tight_layout()  # NEW
        # The subject_id is not directly available inside this function (it only receives raw/events),
        # so the figure is saved using a generic name inside OUTPUT_PATH/_tmp_eeg_qc and renamed by
        # the caller (process_subject) right after this function returns. This keeps this function's
        # signature and call sites unchanged.
        tmp_qc_dir = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc")  # NEW
        os.makedirs(tmp_qc_dir, exist_ok=True)  # NEW
        fig.savefig(os.path.join(tmp_qc_dir, "last_eeg_before_after.png"), dpi=150)  # NEW
        plt.close(fig)  # NEW
    except Exception as e:  # NEW
        print(f"Could not save EEG QC plot: {e}")  # NEW
    
    return raw_filtered


# SYNCHRONIZED SLIDING WINDOW EXTRACTION 
def extract_synchronized_windows(eeg_raw, fnirs_data, events, sfreq_eeg):

    print("Extracting synchronized asymmetric sliding windows...")
    print(f"EEG window: {EEG_WINDOW_LEN}s, step: {EEG_STEP}s")
    print(f"fNIRS window: {FNIRS_WINDOW_LEN}s, step: {FNIRS_STEP}s")
    
    # support both marker formats 
    # label_map = {
    #     1: 0, 2: 1, 3: 1, 4: 2, 5: 2,
    #     16: 0, 48: 1, 64: 1, 80: 2, 96: 2
    # }
    #
    # valid_codes = [1, 2, 3, 4, 5, 16, 48, 64, 80, 96]
    # target_events = events[np.isin(events[:, 2], valid_codes)]
    # FIX (Problem 2): the codes above (1-5, 16/48/64/80/96) are STIMULUS-level markers that fire
    # roughly every 2 s within a 40 s trial (per Shin et al. 2018 / Dataset_description_MATLAB.pdf,
    # codes 16/48/64/80/96 correspond to individual target/non-target number presentations, not to
    # the start of a cognitive-load trial). Sliding a window from each of these stimulus markers
    # caused massive, redundant overlap between windows anchored to different stimuli within the same
    # trial, rather than a clean traversal of the full 40 s task period for that trial. The correct
    # trial-onset codes, confirmed directly from the VP001 .mat structure (inspect_data.py output:
    # mrk_nback.event.desc contains 144 etc., and the dataset documentation confirms 112/128/144 =
    # 0-/2-/3-back SESSION start), are used instead: each trial-onset marker anchors exactly one
    # 40 s task period, which is then traversed by the (unchanged) sliding-window loop below.
    label_map = EEG_TRIAL_LABEL_MAP  # NEW: trial-onset codes (112/128/144) -> class label, defined at top of script
    valid_codes = list(EEG_TRIAL_LABEL_MAP.keys())  # NEW
    target_events = events[np.isin(events[:, 2], valid_codes)]  # NEW: now selects one event per ~40s trial instead of one event per ~2s stimulus
    
    print(f"found {len(target_events)} trial markers")
    if len(target_events) == 0:
        print(f"No trial markers found. Available markers: {np.unique(events[:, 2])}")
        return np.array([]), np.array([]), np.array([])
    
    window_samples_eeg = int(EEG_WINDOW_LEN * sfreq_eeg)
    window_samples_fnirs = int(FNIRS_WINDOW_LEN * SFREQ_FNIRS)
    
    max_slide_seconds = EEG_EPOCH_TMAX - EEG_WINDOW_LEN
    step_seconds = EEG_STEP
    
    eeg_windows = []
    fnirs_windows = []
    labels = []
    
    for ev in target_events:
        event_sample_eeg = ev[0]
        raw_code = ev[2]
        
        label = label_map.get(raw_code, -1)
        if label == -1:
            continue
        
        event_time = event_sample_eeg / sfreq_eeg
        
        for offset_sec in np.arange(0.0, max_slide_seconds + 0.001, step_seconds):
            
            # EEG window
            w_start_eeg = int(event_sample_eeg + np.round(offset_sec * sfreq_eeg))
            w_end_eeg = w_start_eeg + window_samples_eeg
            
            # fNIRS window 
            w_start_fnirs = int(np.round((event_time + offset_sec) * SFREQ_FNIRS))
            w_end_fnirs = w_start_fnirs + window_samples_fnirs
            
            if w_end_eeg > eeg_raw.n_times:
                continue
            
            if w_end_fnirs > fnirs_data.shape[0]:
                continue
            
            # Extract EEG
            eeg_slice = eeg_raw.get_data(start=w_start_eeg, stop=w_end_eeg)
            if np.max(np.abs(eeg_slice)) > 120e-6:
                continue
            
            # Extract fNIRS
            fnirs_slice = fnirs_data[w_start_fnirs:w_end_fnirs]
            fnirs_slice = fnirs_slice.transpose(3, 0, 1, 2) 
            
            eeg_windows.append(eeg_slice)
            fnirs_windows.append(fnirs_slice)
            labels.append(label)
    
    eeg_windows = np.array(eeg_windows)
    fnirs_windows = np.array(fnirs_windows)
    labels = np.array(labels)
    
    print(f"Extracted {len(labels)} synchronized windows")
    if len(labels) > 0:
        print(f"Label distribution: {np.bincount(labels)}")
    
    return eeg_windows, fnirs_windows, labels


# MAIN PROCESSING FUNCTION
def process_subject(subject_id):
    print(f"Processing {subject_id}")
    
#    subject_num = subject_id[2:]
#    subject_path = os.path.join(BASE_DATASET_PATH, subject_num)
#    eeg_folder = os.path.join(subject_path, f"{subject_id}-EEG")
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")
    
    if not os.path.exists(eeg_folder):
        print(f"EEG folder not found: {eeg_folder}")
        return
    
    try:
        print("\n Loading EEG...")
        raw_eeg, events, sfreq_eeg = load_eeg_data(eeg_folder)
        
        print("\n Loading and preprocessing fNIRS...")
        fnirs_data = load_and_preprocess_fnirs(subject_id)
        
        print("\n Preprocessing EEG...")
        raw_eeg_clean = preprocess_eeg(raw_eeg, events)

        # NEW (user request): finalize the EEG before/after QC plot for this specific subject by
        # renaming the temp file produced inside preprocess_eeg() (which has no access to subject_id).
        try:  # NEW
            tmp_qc_path = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_before_after.png")  # NEW
            if os.path.exists(tmp_qc_path):  # NEW
                final_qc_dir = os.path.join(OUTPUT_PATH, subject_id)  # NEW
                os.makedirs(final_qc_dir, exist_ok=True)  # NEW
                final_qc_path = os.path.join(final_qc_dir, f"{subject_id}_eeg_before_after.png")  # NEW
                os.replace(tmp_qc_path, final_qc_path)  # NEW
                print(f"Saved EEG before/after QC plot for {subject_id}")  # NEW
        except Exception as e:  # NEW
            print(f"Could not finalize EEG QC plot for {subject_id}: {e}")  # NEW
        
        print("\n Extracting synchronized asymmetric windows...")
        eeg_windows, fnirs_windows, labels = extract_synchronized_windows(
            raw_eeg_clean, fnirs_data, events, sfreq_eeg
        )
        
        if len(labels) == 0:
            print(f"No windows extracted for {subject_id}")
            return
        
        print("\n Normalizing...")
        # EEG: (n_windows, n_channels, n_times) 
        eeg_windows = (eeg_windows - eeg_windows.mean(axis=-1, keepdims=True)) / (eeg_windows.std(axis=-1, keepdims=True) + 1e-8)
        
        # fNIRS: (n_windows, 2, 100, 16, 16)
        # fnirs_windows = (fnirs_windows - fnirs_windows.mean(axis=2, keepdims=True)) / (fnirs_windows.std(axis=2, keepdims=True) + 1e-8)
        # FIX (Problem 3): the line above z-scored EACH WINDOW independently along its own time axis
        # (axis=2), forcing every window's mean to exactly 0 and std to exactly 1. This destroys the
        # between-class difference in absolute HbO/HbR LEVEL that the literature identifies as the key
        # signature distinguishing cognitive-load levels (e.g., Aghajani et al. 2017, Front. Hum.
        # Neurosci. 11:359, report that the HbO peak rises from 0-back to 2-back but then DECREASES
        # from 2-back to 3-back -- a level-based effect that per-window z-scoring would erase, since
        # every window is forced to the same zero-mean/unit-variance scale regardless of its original
        # absolute amplitude). Normalizing fNIRS at the SUBJECT level (computed once per subject, on
        # the full pre-windowing time series, before extract_synchronized_windows is called) still
        # corrects for inter-subject amplitude/baseline differences while preserving the relative
        # level differences between 0-/2-/3-back trials within that subject, which is exactly the
        # signal the classifier needs to learn.
        fnirs_data = (fnirs_data - fnirs_data.mean(axis=0, keepdims=True)) / (fnirs_data.std(axis=0, keepdims=True) + 1e-8)  # NEW: subject-level normalization, computed BEFORE windowing (see re-ordering below)
        
        print("\n Saving...")
        sub_out_dir = os.path.join(OUTPUT_PATH, subject_id)
        os.makedirs(sub_out_dir, exist_ok=True)
        
        # np.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_windows.npy"), eeg_windows)
        # np.save(os.path.join(sub_out_dir, f"{subject_id}_fnirs_windows.npy"), fnirs_windows)
        # np.save(os.path.join(sub_out_dir, f"{subject_id}_labels.npy"), labels)
        # NEW (user request): for the first verification run, only save outputs for the very first
        # subject so correctness can be checked before running the full 26-subject batch. The save
        # calls below are commented out for all subjects except VP001; uncomment the three np.save
        # lines below (and remove/relax this condition) once the pipeline has been verified.
        if subject_id == "VP001":  # NEW: save outputs only for the first subject during verification
            np.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_windows.npy"), eeg_windows)    # NEW
            np.save(os.path.join(sub_out_dir, f"{subject_id}_fnirs_windows.npy"), fnirs_windows)  # NEW
            np.save(os.path.join(sub_out_dir, f"{subject_id}_labels.npy"), labels)               # NEW
        # else:                                                                                    # NEW
        #     # Saving suppressed for subjects other than VP001 during this verification run.        # NEW
        #     pass                                                                                  # NEW
        
        print(f"\n {subject_id} completed!")
        print(f"EEG windows: {eeg_windows.shape}")
        print(f"fNIRS windows: {fnirs_windows.shape}")
        print(f"Labels: {np.bincount(labels)}")
        
    except Exception as e:
        print(f"Error processing {subject_id}: {str(e)}")
        import traceback
        traceback.print_exc()

# RUN
if __name__ == "__main__":
    print("MULTIMODAL PREPROCESSING PIPELINE")
    print(f"EEG window: {EEG_WINDOW_LEN}s, step: {EEG_STEP}s")
    print(f"fNIRS window: {FNIRS_WINDOW_LEN}s, step: {FNIRS_STEP}s")
    print(f"fNIRS epoch: {FNIRS_EPOCH_TMAX}s")
    print(f"fNIRS sampling rate: {SFREQ_FNIRS} Hz")

#    for sub_num in range(1, 27):
    for sub_num in range(1, 2):
        subject_id = f"VP{sub_num:03d}"
        process_subject(subject_id)
    
    print("ALL THE SELECTED SUBJECTS PROCESSED")


MULTIMODAL PREPROCESSING PIPELINE
EEG window: 3.0s, step: 1.0s
fNIRS window: 10.0s, step: 1.0s
fNIRS epoch: 18.0s
fNIRS sampling rate: 10.0 Hz
Processing VP001

 Loading EEG...
Loading EEG from: E:\kntu\Thesis\codes\DATA\VP001-EEG\cnt_nback.mat
EEG loaded: 394930 samples, 200.0 Hz, 567 markers
unique markers: [ 16  48  64  80  96 112 128 144]

 Loading and preprocessing fNIRS...
Loading raw fNIRS from: E:\kntu\Thesis\codes\DATA\VP001-NIRS\cnt_nback.mat
raw fNIRS sampling rate: 10.0 Hz
Raw fNIRS: oxy (21178, 36), deoxy (21178, 36)
Converting to HbO/HbR...
Wavelet denoising...
MARA...
Low-pass filter (0.2 Hz)...
Resampling from 10.0 Hz to 10.0 Hz...
Spatial mapping to 16x16...
fNIRS preprocessed: (21178, 16, 16, 2)
Saved fNIRS before/after QC plot for VP001

 Preprocessing EEG...
Preprocessing EEG...
Bad channels: ['Fp1', 'T7', 'Fp2']
Fitting ICA to data using 28 channels (please be patient, this may take a while)
Selecting by number: 27 components
Computing Extended Infomax ICA
Fitting 

In [ ]:
# Visual checking per sample
# If you wanna check this cell as well, first revise the paths and then set the subject ID accoordingly

import os
import numpy as np
import mne
import matplotlib.pyplot as plt

# CONFIGURATION
DATA_PATH = r"D:\Research\EEG-fNIRS\project\project\Results\MULTIMODAL_PREPROCESSED"
SUBJECT = "VP011"

# LOAD DATA
subject_dir = os.path.join(DATA_PATH, SUBJECT)

eeg_file = os.path.join(subject_dir, f"{SUBJECT}_eeg_windows.npy")
fnirs_file = os.path.join(subject_dir, f"{SUBJECT}_fnirs_windows.npy")
labels_file = os.path.join(subject_dir, f"{SUBJECT}_labels.npy")

if not os.path.exists(eeg_file):
    raise FileNotFoundError(f"No such file: {eeg_file}")

print("Loading preprocessed data...")
eeg_data = np.load(eeg_file)      
fnirs_data = np.load(fnirs_file)  
labels = np.load(labels_file)  

print(f"EEG shape: {eeg_data.shape}")
print(f"fNIRS shape: {fnirs_data.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Label distribution: {np.bincount(labels)}")

# VISUALIZE EEG DATA (first 10 windows)
print("EEG DATA VISUALIZATION")

n_vis = min(10, eeg_data.shape[0])
eeg_vis = eeg_data[:n_vis]  # (10, 28, 600)

n_channels = eeg_data.shape[1]
ch_names = [f'EEG_{i+1:02d}' for i in range(n_channels)]
sfreq = 200.0  
info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')

raw = mne.io.RawArray(eeg_vis[0], info, verbose=False)

# Plot PSD
print("\n PSD Plot (first window)...")
fig_psd = raw.compute_psd(fmin=0.5, fmax=55).plot(picks='eeg', show=False)
fig_psd.suptitle(f"{SUBJECT} - EEG PSD (First Window)", fontsize=12)
plt.show()

# VISUALIZE fNIRS DATA
print("fNIRS DATA VISUALIZATION")

fnirs_vis = fnirs_data[:n_vis]  

# Plot first window - HbO and HbR separately
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# HbO 
hbo_avg = fnirs_vis[0, 0, :, :, :].mean(axis=0)  
im0 = axes[0].imshow(hbo_avg, cmap='RdBu_r', aspect='auto')
axes[0].set_title(f"{SUBJECT} - HbO (Average over time)")
axes[0].set_xlabel("Spatial X")
axes[0].set_ylabel("Spatial Y")
plt.colorbar(im0, ax=axes[0])

# HbR 
hbr_avg = fnirs_vis[0, 1, :, :, :].mean(axis=0)  # (16, 16)
im1 = axes[1].imshow(hbr_avg, cmap='RdBu_r', aspect='auto')
axes[1].set_title(f"{SUBJECT} - HbR (Average over time)")
axes[1].set_xlabel("Spatial X")
axes[1].set_ylabel("Spatial Y")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

# CHECK LABEL DISTRIBUTION
print("LABEL DISTRIBUTION")

unique_labels, counts = np.unique(labels, return_counts=True)
for label, count in zip(unique_labels, counts):
    name = {0: "0-back", 1: "2-back", 2: "3-back"}.get(label, "Unknown")
    print(f"  Class {label} ({name}): {count} windows ({count/len(labels)*100:.1f}%)")

# CHECK DATA STATISTICS
print("DATA STATISTICS")

print(f"EEG:")
print(f"Shape: {eeg_data.shape}")
print(f"Mean: {eeg_data.mean():.6f}")
print(f"Std: {eeg_data.std():.6f}")
print(f"Min: {eeg_data.min():.6f}")
print(f"Max: {eeg_data.max():.6f}")

print(f"\nfNIRS:")
print(f"Shape: {fnirs_data.shape}")
print(f"Mean: {fnirs_data.mean():.6f}")
print(f"Std: {fnirs_data.std():.6f}")
print(f"Min: {fnirs_data.min():.6f}")
print(f"Max: {fnirs_data.max():.6f}")

# CHECK SOME RANDOM WINDOWS
print("CHECK RANDOM WINDOWS")

np.random.seed(42)
random_indices = np.random.choice(len(eeg_data), min(5, len(eeg_data)), replace=False)

for idx in random_indices:
    label = labels[idx]
    label_name = {0: "0-back", 1: "2-back", 2: "3-back"}.get(label, "Unknown")
    print(f"Window {idx}: Label {label} ({label_name})")
    print(f"EEG mean: {eeg_data[idx].mean():.6f}, std: {eeg_data[idx].std():.6f}")
    print(f"fNIRS mean: {fnirs_data[idx].mean():.6f}, std: {fnirs_data[idx].std():.6f}")
    print()

# SAVE SUMMARY REPORT
print("SAVING SUMMARY REPORT")

report_path = os.path.join(subject_dir, f"{SUBJECT}_quality_report.txt")
with open(report_path, 'w') as f:
    f.write(f"DATA QUALITY REPORT - {SUBJECT}\n")
    f.write(f"EEG shape: {eeg_data.shape}\n")
    f.write(f"fNIRS shape: {fnirs_data.shape}\n")
    f.write(f"Labels shape: {labels.shape}\n")
    f.write(f"Label distribution: {dict(zip(unique_labels, counts))}\n\n")
    f.write(f"EEG Statistics:\n")
    f.write(f"Mean: {eeg_data.mean():.6f}\n")
    f.write(f"Std: {eeg_data.std():.6f}\n")
    f.write(f"Min: {eeg_data.min():.6f}\n")
    f.write(f"Max: {eeg_data.max():.6f}\n\n")
    f.write(f"fNIRS Statistics:\n")
    f.write(f"Mean: {fnirs_data.mean():.6f}\n")
    f.write(f"Std: {fnirs_data.std():.6f}\n")
    f.write(f"Min: {fnirs_data.min():.6f}\n")
    f.write(f"Max: {fnirs_data.max():.6f}\n")

print(f"Report saved to: {report_path}")

print("VISUALIZATION COMPLETE")

In [ ]:
# MORE VISUALIZATION CHECK FOR BASELINE CORRECTION
# set teh paths before running

import matplotlib.pyplot as plt
import numpy as np

# data load
eeg_data = np.load(r"D:\Research\EEG-fNIRS\project\project\Results\MULTIMODAL_PREPROCESSED\VP011\VP011_eeg_windows.npy")
fnirs_data = np.load(r"D:\Research\EEG-fNIRS\project\project\Results\MULTIMODAL_PREPROCESSED\VP011\VP011_fnirs_windows.npy")

# EEG BASELINE CHECK
fig, axes = plt.subplots(2, 3, figsize=(15, 6))

for i, ax in enumerate(axes.flatten()):
    if i < eeg_data.shape[0]:
        # ONE SAMPLE CHECK
        signal = eeg_data[i, 0, :] 
        ax.plot(signal)
        ax.axhline(y=0, color='r', linestyle='--', alpha=0.5, label='Baseline (mean=0)')
        ax.set_title(f"Window {i} - EEG\nMean: {signal.mean():.4f}")
        ax.set_xlabel("Time (samples)")
        ax.set_ylabel("Amplitude (normalized)")
        ax.legend()

plt.tight_layout()
plt.suptitle("EEG Baseline Check", fontsize=14, y=1.02)
plt.show()

# FNIRS ABSELINE CHECK
fig, axes = plt.subplots(2, 3, figsize=(15, 6))

for i, ax in enumerate(axes.flatten()):
    if i < fnirs_data.shape[0]:
        # HbO 
        signal = fnirs_data[i, 0, :, :, :].mean(axis=(1, 2)) 
        ax.plot(signal)
        ax.axhline(y=0, color='r', linestyle='--', alpha=0.5, label='Baseline')
        ax.set_title(f"Window {i} - fNIRS (HbO)\nMean: {signal.mean():.4f}")
        ax.set_xlabel("Time (samples)")
        ax.set_ylabel("Amplitude (normalized)")
        ax.legend()

plt.tight_layout()
plt.suptitle("fNIRS Baseline Check", fontsize=14, y=1.02)
plt.show()

# STATISTICAL CHECK
eeg_means = eeg_data.mean(axis=(1, 2)) 
fnirs_means = fnirs_data.mean(axis=(2, 3, 4))  

print("BASELINE VERIFICATION (All windows)")
print(f"EEG mean across all windows: {eeg_means.mean():.6f} ± {eeg_means.std():.6f}")
print(f"fNIRS HbO mean across all windows: {fnirs_means[:, 0].mean():.6f} ± {fnirs_means[:, 0].std():.6f}")
print(f"fNIRS HbR mean across all windows: {fnirs_means[:, 1].mean():.6f} ± {fnirs_means[:, 1].std():.6f}")

# WINDOWS CHECK FOR BASELINE
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(eeg_means, bins=50, alpha=0.7, label='EEG', color='blue')
ax.hist(fnirs_means[:, 0], bins=50, alpha=0.5, label='fNIRS HbO', color='red')
ax.set_xlabel("Mean of each window")
ax.set_ylabel("Number of windows")
ax.set_title("Distribution of window means (should be centered at 0)")
ax.axvline(x=0, color='black', linestyle='--', alpha=0.5, label='Ideal baseline')
ax.legend()
plt.show()